In [3]:
from collections import defaultdict
from itertools import product
import os
from pathlib import Path
import re

os.environ["OPENCV_IO_ENABLE_OPENEXR"]="1"

from astropy.io import fits
from astropy.table import Table
import cv2
from cytoolz import groupby
from dustgoggles.structures import NestingDict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sh
import skimage.io

from asdf.asdf_utils import cast_to_reference
from asdf.scan import cluster_observations, scan_zcam_files
from asdf.zcam_bandset import ZcamBandSet
from asdf_settings.rapidlooks import CROP_SETTINGS
from marslab.compat.xcam import DERIVED_CAM_DICT
from marslab.imgops.imgutils import crop, normalize_range

%matplotlib qt

In [403]:
clusters = cluster_observations(
    scan_zcam_files('/datascratch/zcam_data/products/0083/iof')
)[0]
len(clusters)

5

In [404]:
bandsets = [ZcamBandSet(c) for c in clusters.values()]
seqgroups = groupby(lambda bs: bs.metadata['SEQ_ID'].iloc[0], bandsets)
mosaics = {k: v for k, v in seqgroups.items() if len(v) > 1}
mosaic = list(mosaics.values())[0]

In [406]:
outpath = Path("/datascratch/zcam_data/hugin_test_dump")
outpath.mkdir(exist_ok=True)
bands = mosaic[0].metadata['BAND'].unique()
tiff_info = []
for bandset, band in product(mosaic, bands):
    bandset.load([band])
    bandset.bulk_debayer([band])
    cropped = crop(bandset.get_band(band), CROP_SETTINGS['crop'])
    tiff_path = Path(outpath, f"{band}_{bandset.name}.tiff")
    # note: hugin crashes if the input extension is "tif", but insists on writing it as "tiff".
    # it also exhibits different behavior based on tiff file data type.
    # also note that cv2 will write but not read 32-bit tiff files, and
    # pillow won't do either. scikit-image can read them.
    cv2.imwrite(str(tiff_path), cropped)
    # for -f argument to pto_gen 
    azimuth_fov = bandset.precached[
        bandset.metadata.loc[bandset.metadata['BAND'] == band, 'PATH'].iloc[0]
    ].metaget("AZIMUTH_FOV")['value']
    tiff_rec = {
        'band': band,
        'rsm': bandset.metadata['RSM'].iloc[0],
        'seq_id': bandset.metadata['SEQ_ID'].iloc[0],
        'bandset_name': bandset.name,
        'path': tiff_path, 
        'fov': azimuth_fov, 
        'eye': band[0],
    }
    tiff_info.append(tiff_rec)
    bandset.purge()
tiff_info = pd.DataFrame(tiff_info)

In [13]:
def zcam_pto_gen(paths, azimuth_fov, output_file=None):
    if output_file is None:
        output_file = Path(Path(paths[0]).parent, Path(paths[0]).stem + ".pto")
    return sh.pto_gen(
        *("-o", str(output_file)), 
        *("-p", 0),
        *("-f", azimuth_fov), 
        *("-s", len(paths)),
        "--ignore-fov-rectilinear",
        *tuple(map(str, paths))
    ), output_file

# they really like to put it in there
def remove_hugin_crop_instruction(pto_file):
    with open(pto_file) as stream:
        text = stream.read()
    dimension_match = re.search(r'f\d+ w(\d+) h(\d+)', text)
    width, height = dimension_match.group(1), dimension_match.group(2)
    crop_match = re.search(r'k\d+ E\d+ R\d+ (S(\d|,)+) n"TIFF', text)
    text = text.replace(crop_match.group(1), f"S0,{width},0,{height}")
    with open(pto_file, "w") as stream:
        stream.write(text)
    return width, height
        
def hugin_assistant(pto_file):
    return sh.hugin_executor("--assistant", pto_file)
        
def execute_hugin_stitch(pto_file, prefix=None):
    command_parts = ['-s', pto_file]
    if prefix is not None:
        command_parts = ['-p', prefix] + command_parts
    return sh.hugin_executor(*command_parts)

def read_first_channel(path):
    return skimage.io.imread(path)[:, :, 0]

In [135]:
def pano_modify(
    pto_file, 
    output_type="NORMAL,BF,REMAPORIG",
    # projection: 0=equirectangular, 1=cylindrical, ...?
    # canvas: AUTO
    projection=1,
    **kwargs
):
    return sh.pano_modify(
        pto_file,
        output=pto_file,
        projection=projection,
        output_type=output_type,
        **kwargs
    )

def crop_outer(array):
    nzy, nzx = np.nonzero(array)
    return crop(array, (nzx.min(), nzx.max(), nzy.min(), nzy.max()))


def make_eye_mosaics(eye, tiff_info):
    eye_name = {'L': 'left', 'R': 'right'}[eye]
    available_bands = tiff_info['band'].tolist()
    eye_bands = [b for b in available_bands if b.startswith(eye)]
    if len(eye_bands) == 0:
        return {}, None
    ref_band = next(
        (f"{eye}{n}" for n in PREFERRED_REF_BANDS if f"{eye}{n}" in eye_bands)
    )
    ref_slice = tiff_info.loc[tiff_info['band'] == ref_band]
    ref_paths, ref_fovs = ref_slice['path'].tolist(), ref_slice['fov']
    print(f"referencing {eye_name} mosaics to {ref_band}")
    _, ref_pto_file = zcam_pto_gen(ref_paths, float(np.mean(ref_fovs)))
    hugin_assistant(ref_pto_file)
    pano_modify(ref_pto_file, canvas="AUTO")
    remove_hugin_crop_instruction(ref_pto_file)
    with open(ref_pto_file) as stream:
        ref_text = stream.read()
    pto_files = {ref_band: ref_pto_file}
    for band in filter(lambda b: b != ref_band, eye_bands):
        band_slice = tiff_info.loc[tiff_info['band'] == band]
        if len(band_slice) != len(ref_slice):
            raise ValueError('mismatched availability between bands.')
        pto_file = Path(
            ref_pto_file.parent, ref_pto_file.name.replace(ref_band, band)
        )
        band_text = ref_text
        for _, row in ref_slice.iterrows():
            ref_path = row['path']
            band_path = band_slice.loc[band_slice['rsm'] == row['rsm'], 'path'].iloc[0]
            band_text = band_text.replace(ref_path.name, band_path.name)
        with pto_file.open("w") as stream:
            stream.write(band_text)
        pto_files[band] = pto_file
    for band, pto_file in pto_files.items():
        print(f"generating {band} mosaics")
        execute_hugin_stitch(pto_file)
    return pto_files, ref_text

In [21]:
PREFERRED_REF_BANDS = (1, 2, 3, 4, 5, 6, "0R", "0G", "0B")
process_info = {}
for eye in ("L", "R"):
    process_info[eye] = make_eye_mosaics(eye, tiff_info)

referencing left mosaics to L1
generating L1 mosaics
generating L0B mosaics
generating L0G mosaics
generating L0R mosaics
generating L2 mosaics
generating L3 mosaics
generating L4 mosaics
generating L5 mosaics
generating L6 mosaics
referencing right mosaics to R1
generating R1 mosaics
generating R0B mosaics
generating R0G mosaics
generating R0R mosaics
generating R2 mosaics
generating R3 mosaics
generating R4 mosaics
generating R5 mosaics
generating R6 mosaics


In [276]:
LONG_METADATA_DTYPES = {
    "SOL": "int16",
    "WAVELENGTH": "float16",
    "IX": "uint8",
    "SOLAR_ELEVATION": "float32",
    "INSTRUMENT_ELEVATION": "float32",
    "L_S": "float32",
    "INSTRUMENT_AZIMUTH": "float32",
    "SOLAR_AZIMUTH": "float32",
    "SCLK": "float64",
    "SITE": "int16",
    "DRIVE": "int16",
    "CTIME": "int64",
    "ZOOM": "uint8",
    "VERSION": "uint8",
    "RSM": "int16",
    "CALTARGET_LTST": "float32",
    "SUBFRAME": str,
    "MINI_HEADER": str,
    "RMC": str,
    "BAYER_PIXEL": str
}
all_metadata = pd.concat([b.metadata for b in bandsets])
all_metadata = all_metadata.drop(columns=['ANALYSIS_NAME', 'stem', 'PATH'])
all_metadata = cast_to_reference(all_metadata, LONG_METADATA_DTYPES)

In [277]:
eye_pto_files, eye_ref_text = process_info['L']
paths = {
    band: Path(p.parent, p.name.replace(".pto", ".tif")) for 
    band, p in eye_pto_files.items()
}
arrays = {
    band: crop_outer(read_first_channel(phot)) 
    for band, phot in paths.items()
}
if not len({arr.shape for arr in arrays.values()}) == 1:
    raise ValueError("apparent misalignment.")

In [278]:
hdus = [fits.PrimaryHDU()]
hdus += [
    fits.ImageHDU(phot_arrays[band], name=band) 
    for band in sorted(phot_arrays.keys())
]
meta_hdu = fits.table_to_hdu(
    Table.from_pandas(all_metadata.loc[all_metadata['BAND'].isin(arrays.keys())])
)
meta_hdu.name = 'metadata'
hdus.append(meta_hdu)

In [280]:
mosaic_fn = (
    f"SOL{meta_hdu.data['SOL'][0]}_"
    f"{meta_hdu.data['SEQ_ID'][0]}_"
    f"{meta_hdu.data['BAND'][0][0]}_mosaic.fits"
)

In [281]:
hdul = fits.HDUList(hdus)

In [283]:
hdul.writeto(Path(Path(list(eye_pto_files.values())[0]).parent, mosaic_fn))

In [122]:
from marslab.imgops.loaders import pdr_load, pil_load_shell

In [256]:
mosaic_fits = "/datascratch/zcam_data/hugin_test_dump/SOL604_ZCAM03467_L_mosaic.fits"

In [358]:
def simple_fits_load(path, metadata, bands, precached=None):
    from astropy.io import fits
    if precached is not None:
        hdul = fits.open(path)
    else:
        hdul = precached
    arrays = {}
    for band in bands:
        arrays[band] = (
            hdul[metadata.loc[metadata['BAND'] == band, 'IX'].iloc[0]].data
        )
    return arrays

In [376]:
class ZcamMosaicBandSet(BandSet):
    def __init__(self, mosaic_fits):
        # TODO, maybe: assess whether this holds too much stuff in memory
        self.precached = fits.open(mosaic_fits)
        band_hdu = {
            info[1]: info[0] for info in mhdul.info(False)
            if info[3] == 'ImageHDU'
        }
        metadata_shell = pd.DataFrame(
            {
                'BAND': list(band_hdu.keys()), 
                'IX': list(band_hdu.values()), 
                'PATH': mosaic_fits
            }
        )
        metadata_shell['WAVELENGTH'] = [
            DERIVED_CAM_DICT['ZCAM']['filters'][band] 
            for band in metadata_shell['BAND']
        ]
        metadata_records = fits.open(mosaic_fits)[-1].data
#         metadata = pd.DataFrame(enforce_order_and_object(metadata_records))
#         metadata = metadata.drop(columns=['BAND', 'IX'])
#         for c in metadata.columns:
#             if isinstance(metadata[c].iloc[0], bytes):
#                 metadata[c] = metadata[c].map(lambda b: b.decode('utf-8'))
#         metadata = pd.concat([metadata_shell, metadata], axis=1)
        super().__init__(metadata=metadata_shell, load_method=simple_fits_load)


In [377]:
mbandset = ZcamMosaicBandSet(mosaic_fits)

In [ ]:
from asdf_settings.generators.look_assembler import RAPIDLOOKS

In [399]:
look = RAPIDLOOKS[16]
print(look['name'])
if 'crop' in look.keys():
    del look['crop']

dcs L2_L5_L6


In [400]:
mbandset.load('all')
mbandset.make_look_set([look])

In [393]:
mbandset.looks

{'band_depth L4_L3_L2 masked': <Figure size 1920x1041 with 2 Axes>}

In [ ]:
phots = [
    p for p in tuple(Path('/datascratch/zcam_data/hugin_test_dump/').iterdir())
    if p.suffix == ".tif" and "fused" not in p.name and "layers" not in p.name
]

In [ ]:
phots

In [ ]:
he_cmd = execute_hugin_stitch(new_pto_file)

In [ ]:
he_cmd = execute_hugin_stitch(pto_file)

In [ ]:
eye = 'L'
eye_phots = [p for p in phots if p.name.startswith(eye)]

In [ ]:
eyehash = {
#     p.name.split("_")[0]: crop_outer(read_first_channel(p)) for p in eye_phots
    p.name.split("_")[0]: read_first_channel(p) for p in eye_phots
}

In [ ]:
{filt: arr.shape for filt, arr in eyehash.items()}

In [ ]:
stak = np.dstack([eyehash['L0R'], eyehash['L5'], eyehash['L6']])

In [ ]:
plt.imshow(eyehash['L6'])

In [ ]:
plt.imshow(stak)

In [ ]:
nophot_path = Path(pto_file.parent, f'{pto_file.stem}_fused.tif')
phot_path = Path(pto_file.parent, f'{pto_file.stem}.tif')

In [ ]:
nophot = crop_outer(read_first_channel(nophot_path))

In [ ]:
plt.imshow(nophot_crop)

In [ ]:
plane = read_first_plane(nophot_path)

In [ ]:
plt.imshow(im[:, :, 0])

In [ ]:
plt.imshow(im)

In [ ]:
im

In [ ]:
nophot.shape

In [ ]:
nophot

In [ ]:
nophot.max()

In [ ]:
np.asanyarray(Image.open('/datascratch/zcam_data/hugin_test_dump/L2_SOL0604_zcam03468_RSM742.tiff')).dtype

In [ ]:
(nophot[:, :, 0] == nophot[:, :, 1]).all()

In [ ]:
def outer_crop(image)

In [ ]:
# def cpfind(pto_file, output_file=None):
#     if output_file is None:
#         output_file = Path(
#             Path(pto_file).parent, Path(pto_file).stem + "_cp.pto"
#         )
#     return sh.cpfind("-o", output_file, pto_file), output_file

# def autooptimiser(pto_file, output_file=None):
#     if output_file is None:
#         output_file = Path(
#             Path(pto_file).parent, Path(pto_file).stem + "_ao.pto"
#         )
#     return sh.autooptimiser("-a", "-p", "-o", output_file, pto_file), output_file